In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import optuna
from optuna.samplers import TPESampler
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns

# NEW: For Neural Network
import torch
import torch.nn as nn
import torch.optim as optim
from skorch import NeuralNetRegressor


# Set a random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED) # For PyTorch reproducibility

print("Starting Kaggle House Price Prediction Script...")

# Step 1: Load data
print("Loading training and test data...")
TRAIN_PATH = '/kaggle/input/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '/kaggle/input/house-prices-advanced-regression-techniques/test.csv'

try:
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    print("Data loaded successfully from Kaggle paths.")
except FileNotFoundError:
    print(f"Error: Make sure '{TRAIN_PATH}' and '{TEST_PATH}' are available in your Kaggle environment.")
    print("Double-check the dataset links or input paths in your Kaggle notebook.")
    exit()

# Step 2: Load external data (example - adapt to your source)
print("No external data loaded for this run. If you have external data, uncomment and adapt the code in Step 2.")

# --- NEW: Step 2.5: Explicit Outlier Treatment with Visualizations ---
print("\nDetecting and treating outliers with visualizations...")

# Visualize outliers in GrLivArea and LotArea before removal
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.boxplot(y=train['GrLivArea'])
plt.title('Box Plot of GrLivArea (Before Outlier Treatment)')
plt.ylabel('GrLivArea')

plt.subplot(1, 2, 2)
sns.scatterplot(x=train['GrLivArea'], y=train['SalePrice'])
plt.title('Scatter Plot of GrLivArea vs SalePrice (Before Outlier Treatment)')
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')
plt.tight_layout()
plt.show()


print("Identified extreme outliers in GrLivArea vs SalePrice. Removing or capping them.")
# Common outliers in this dataset (GrLivArea > 4000, usually with low SalePrice)
initial_train_shape = train.shape[0]
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)
print(f"Removed {initial_train_shape - train.shape[0]} rows based on GrLivArea and SalePrice outlier criteria.")

# Re-visualize after treatment (optional, but good for verification)
# plt.figure(figsize=(7, 6))
# sns.boxplot(y=train['GrLivArea'])
# plt.title('Box Plot of GrLivArea (After Outlier Treatment)')
# plt.ylabel('GrLivArea')
# plt.show()
# 

print("Outlier treatment completed for GrLivArea. Other features may be considered.")
# --- END NEW: Step 2.5 ---


# Step 3: More Comprehensive Missing Value Imputation based on Data Description
print("\nPerforming advanced missing value imputation...")
cat_none_features = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
]
for col in cat_none_features:
    if col in train.columns:
        train[col] = train[col].fillna('None')
    if col in test.columns:
        test[col] = test[col].fillna('None')

num_zero_features = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea', 'MasVnrArea',
    'GarageYrBlt', 'LotFrontage' 
]
for col in num_zero_features:
    if col in train.columns:
        train[col] = train[col].fillna(0)
    if col in test.columns:
        test[col] = test[col].fillna(0)

if 'Electrical' in train.columns:
    train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])
if 'Electrical' in test.columns:
    test['Electrical'] = test['Electrical'].fillna(test['Electrical'].mode()[0])

print("Missing value imputation completed.")


# Step 4: Advanced Feature Engineering - Heart-winning Features!
print("\nPerforming advanced feature engineering with deep data understanding...")
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']
train['TotalBath'] = train['FullBath'] + 0.5 * train['HalfBath'] + train['BsmtFullBath'] + 0.5 * train['BsmtHalfBath']
test['TotalBath'] = test['FullBath'] + 0.5 * test['HalfBath'] + test['BsmtFullBath'] + 0.5 * test['BsmtHalfBath']
train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']
train['YearsSinceRemodel'] = train['YrSold'] - train['YearRemodAdd']
test['YearsSinceRemodel'] = test['YrSold'] - test['YearRemodAdd']
train['YearsSinceRemodel'] = train['YearsSinceRemodel'].apply(lambda x: max(0, x))
test['YearsSinceRemodel'] = test['YearsSinceRemodel'].apply(lambda x: max(0, x))
train['HasPool'] = train['PoolQC'].apply(lambda x: 1 if x != 'None' else 0)
test['HasPool'] = test['PoolQC'].apply(lambda x: 1 if x != 'None' else 0)
train['HasGarage'] = train['GarageType'].apply(lambda x: 1 if x != 'None' else 0)
test['HasGarage'] = test['GarageType'].apply(lambda x: 1 if x != 'None' else 0)
train['HasBsmt'] = train['BsmtQual'].apply(lambda x: 1 if x != 'None' else 0)
test['HasBsmt'] = test['BsmtQual'].apply(lambda x: 1 if x != 'None' else 0)
train['HasFireplace'] = train['FireplaceQu'].apply(lambda x: 1 if x != 'None' else 0)
test['HasFireplace'] = test['FireplaceQu'].apply(lambda x: 1 if x != 'None' else 0)
train['TotalPorchSF'] = train['OpenPorchSF'] + train['EnclosedPorch'] + train['3SsnPorch'] + train['ScreenPorch']
test['TotalPorchSF'] = test['OpenPorchSF'] + test['EnclosedPorch'] + test['3SsnPorch'] + test['ScreenPorch']
train['OverallQual_GrLivArea'] = train['OverallQual'] * train['GrLivArea']
test['OverallQual_GrLivArea'] = test['OverallQual'] * test['GrLivArea']
train['OverallQual_TotalSF'] = train['OverallQual'] * train['TotalSF']
test['OverallQual_TotalSF'] = test['OverallQual'] * test['TotalSF']
train['LivingAreaToLotRatio'] = train['GrLivArea'] / train['LotArea'].replace(0, np.nan)
test['LivingAreaToLotRatio'] = test['GrLivArea'] / test['LotArea'].replace(0, np.nan)
train['LivingAreaToLotRatio'] = train['LivingAreaToLotRatio'].fillna(0)
test['LivingAreaToLotRatio'] = test['LivingAreaToLotRatio'].fillna(0)

quality_map_num = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols_to_map = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]
for df in [train, test]:
    df['Quality_Score'] = df['OverallQual'].fillna(0) + df['OverallCond'].fillna(0)
    for col in qual_cols_to_map:
        if col in df.columns:
            df['Quality_Score'] += df[col].map(quality_map_num).fillna(0)
train['Total_Living_Rooms'] = train['TotRmsAbvGrd'] + train['FullBath'] + 0.5 * train['HalfBath']
test['Total_Living_Rooms'] = test['TotRmsAbvGrd'] + test['FullBath'] + 0.5 * test['HalfBath']
train['Effective_Age'] = train[['HouseAge', 'YearsSinceRemodel']].min(axis=1)
test['Effective_Age'] = test[['HouseAge', 'YearsSinceRemodel']].min(axis=1)
train['IsRecentBuild'] = (train['YearBuilt'] >= 2000).astype(int)
test['IsRecentBuild'] = (test['YearBuilt'] >= 2000).astype(int)
train['HasCentralAir'] = (train['CentralAir'] == 'Y').astype(int)
test['HasCentralAir'] = (test['CentralAir'] == 'Y').astype(int)

# --- MORE ADVANCED FEATURE ENGINEERING (Inspired by FE notebooks) ---

# 1. Total Square Feet per Room (Density)
train['TtlSF_per_Room'] = train['TotalSF'] / train['TotRmsAbvGrd'].replace(0, 1) # Avoid division by zero
test['TtlSF_per_Room'] = test['TotalSF'] / test['TotRmsAbvGrd'].replace(0, 1)

# 2. Garage Quality and Area interaction
train['GarageQual_Area'] = train['GarageQual'].map(quality_map_num) * train['GarageArea']
test['GarageQual_Area'] = test['GarageQual'].map(quality_map_num) * test['GarageArea']
# Fill NaN from map (for 'None' garage types that become 0)
train['GarageQual_Area'] = train['GarageQual_Area'].fillna(0)
test['GarageQual_Area'] = test['GarageQual_Area'].fillna(0)

# 3. Year built and remodel interaction (reflects significant updates)
train['YearBuilt_Remod'] = train['YearBuilt'] * train['YearRemodAdd']
test['YearBuilt_Remod'] = test['YearBuilt'] * test['YearRemodAdd']

# 4. Binary feature for whether house was remodeled (if YearRemodAdd != YearBuilt)
train['IsRemodeled'] = (train['YearRemodAdd'] != train['YearBuilt']).astype(int)
test['IsRemodeled'] = (test['YearRemodAdd'] != test['YearBuilt']).astype(int)

# 5. Polynomial features for key numerical columns
# Note: These will be added directly to the DataFrame. Their scaling will be handled by RobustScaler later.
# It's good practice to apply these to features that have a strong, non-linear relationship with target.
for col in ['GrLivArea', 'TotalSF', 'LotArea', 'OverallQual', 'HouseAge', 'TotalBath', 'Quality_Score']:
    if col in train.columns:
        # Check if the polynomial feature already exists to prevent duplicate creation
        poly_col_name = f'{col}_sq'
        if poly_col_name not in train.columns:
            train[poly_col_name] = train[col]**2
            test[poly_col_name] = test[col]**2


print("Feature engineering completed with deep data understanding.")

# Log-transform highly skewed numerical features
# Ensure newly created numerical features are also considered for log-transformation if skewed
skewed_features.extend([
    'TtlSF_per_Room', 'GarageQual_Area', 'YearBuilt_Remod',
    # Add polynomial features generated above if they are new and numerical
    col + '_sq' for col in ['GrLivArea', 'TotalSF', 'LotArea', 'OverallQual', 'HouseAge', 'TotalBath', 'Quality_Score'] if f'{col}_sq' in train.columns and f'{col}_sq' not in skewed_features
])
# Remove duplicates and ensure only numeric features are included
skewed_features = list(set(skewed_features))
skewed_features = [f for f in skewed_features if f in train.columns and np.issubdtype(train[f].dtype, np.number)]


for col in skewed_features:
    if col in train.columns:
        train[col] = np.log1p(train[col].fillna(0))
    if col in test.columns:
        test[col] = np.log1p(test[col].fillna(0))


# Step 5: Define features (X) and target (y) AFTER ALL FEATURE ENGINEERING and Outlier Treatment
train_ids = train['Id']
test_ids = test['Id']

X = train.drop(['Id', 'SalePrice'], axis=1)
y = np.log1p(train['SalePrice'])

test = test.drop('Id', axis=1)

# Align columns
train_cols = X.columns
test_cols = test.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    test[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X[c] = 0

test = test[train_cols]


# Step 6: Identify numerical, ordinal, and nominal categorical columns *AFTER* feature engineering
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()

ordinal_features_mapping = {
    'LotShape': ['None', 'IR3', 'IR2', 'IR1', 'Reg'],
    'Utilities': ['None', 'NoSeWa', 'NoSewr', 'AllPub'],
    'LandContour': ['None', 'Low', 'Bnk', 'HLS', 'Lvl'],
    'ExterQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence': ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'PavedDrive': ['N', 'P', 'Y'],
    'OverallQual': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'OverallCond': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
}

for col, order in ordinal_features_mapping.items():
    if col in X.columns and 'None' not in order and 'None' in X[col].unique():
        order.insert(0, 'None')

ordinal_cols = list(ordinal_features_mapping.keys())
nominal_cols = [col for col in X.select_dtypes(include='object').columns.tolist() if col not in ordinal_cols]

newly_engineered_binary_features = ['IsRecentBuild', 'HasCentralAir', 'IsRemodeled'] # Added IsRemodeled
for col in newly_engineered_binary_features:
    if col in X.columns and col not in numerical_cols:
        numerical_cols.append(col)

# Ensure newly engineered numerical features are in numerical_cols
newly_engineered_numerical_features = [
    'Quality_Score', 'Total_Living_Rooms', 'Effective_Age',
    'TtlSF_per_Room', 'GarageQual_Area', 'YearBuilt_Remod'
]
newly_engineered_numerical_features.extend([f'{col}_sq' for col in ['GrLivArea', 'TotalSF', 'LotArea', 'OverallQual', 'HouseAge', 'TotalBath', 'Quality_Score']])
for col in newly_engineered_numerical_features:
    if col in X.columns and col not in numerical_cols:
        numerical_cols.append(col)

numerical_cols = [col for col in numerical_cols if col not in ordinal_cols and col not in nominal_cols]


# --- HELPER FUNCTION for K-Fold Target Encoding ---
def apply_kfold_target_encoding(X_data, y_data=None, X_apply=None, n_splits=5, target_col='Neighborhood', smooth_factor=1.0):
    X_data_copy = X_data.copy()
    X_apply_copy = X_apply.copy() if X_apply is not None else None

    oof_te_values = np.zeros(len(X_data_copy))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    
    global_mean = y_data.mean() if y_data is not None else 0

    for tr_idx, va_idx in kf.split(X_data_copy):
        X_tr, X_va = X_data_copy.iloc[tr_idx], X_data_copy.iloc[va_idx]
        y_tr, _ = y_data.iloc[tr_idx], y_data.iloc[va_idx]

        mean_te = y_tr.groupby(X_tr[target_col]).mean()
        
        counts = X_tr[target_col].value_counts()
        smoothed_mean_te = (mean_te * counts + global_mean * smooth_factor) / (counts + smooth_factor)

        oof_te_values[va_idx] = X_va[target_col].map(smoothed_mean_te).fillna(global_mean)

    X_data_copy[f'{target_col}_AvgPrice'] = oof_te_values
    X_data_copy[f'{target_col}_AvgPrice'] = np.log1p(X_data_copy[f'{target_col}_AvgPrice'])

    if X_apply_copy is not None:
        full_data_mean_te = y_data.groupby(X_data_copy[target_col]).mean()
        full_data_counts = X_data_copy[target_col].value_counts()
        full_data_smoothed_mean_te = (full_data_mean_te * full_data_counts + global_mean * smooth_factor) / (full_data_counts + smooth_factor)
        
        X_apply_copy[f'{target_col}_AvgPrice'] = X_apply_copy[target_col].map(full_data_smoothed_mean_te).fillna(global_mean)
        X_apply_copy[f'{target_col}_AvgPrice'] = np.log1p(X_apply_copy[f'{target_col}_AvgPrice'])
        return X_apply_copy
    
    return X_data_copy

# Add the new target-encoded feature to numerical_cols for the preprocessor
if 'Neighborhood' in X.columns and 'Neighborhood_AvgPrice' not in numerical_cols:
    numerical_cols.append('Neighborhood_AvgPrice')


# Step 7: Preprocessing pipeline updated for ordinal features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', RobustScaler())
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('ordinal', OrdinalEncoder(categories=[ordinal_features_mapping[col] for col in ordinal_cols],
                               handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_template = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('ord', ordinal_transformer, ordinal_cols),
        ('cat', categorical_transformer, nominal_cols)
    ],
    remainder='passthrough'
)


# Step 8: Hyperparameter tuning for XGBoost with Optuna
print("\nTuning XGBoost hyperparameters with Optuna...")
def objective_xgb(trial):
    n_estimators = trial.suggest_int('n_estimators', 200, 1500, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.15, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
    gamma = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True)
    
    params = {
        'n_estimators': n_estimators, 'learning_rate': learning_rate, 'max_depth': max_depth,
        'subsample': subsample, 'colsample_bytree': colsample_bytree, 'gamma': gamma,
        'reg_alpha': reg_alpha, 'reg_lambda': reg_lambda,
        'random_state': RANDOM_SEED, 'n_jobs': -1
    }
    model = xgb.XGBRegressor(**params, eval_metric='rmse', early_stopping_rounds=50)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    rmse_scores = []
    
    for train_idx, val_idx in cv.split(X):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        X_train_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, target_col='Neighborhood')
        X_val_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, X_val_fold, target_col='Neighborhood')

        local_preprocessor = preprocessor_template
        local_preprocessor.fit(X_train_fold_te)

        X_train_transformed = local_preprocessor.transform(X_train_fold_te)
        X_val_transformed = local_preprocessor.transform(X_val_fold_te)

        model.fit(X_train_transformed, y_train_fold, eval_set=[(X_val_transformed, y_val_fold)], verbose=False)

        y_pred = model.predict(X_val_transformed)
        rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))
        rmse_scores.append(rmse)
    return np.mean(rmse_scores)

print("Creating Optuna study for XGBoost with TPESampler...")
study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_SEED))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"Final best XGBoost parameters: {best_xgb_params}")

# Step 9: Hyperparameter tuning for LightGBM with Optuna
print("\nTuning LightGBM hyperparameters with Optuna...")
def objective_lgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 200, 1500, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.15, log=True)
    num_leaves = trial.suggest_int('num_leaves', 20, 60)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_child_samples = trial.suggest_int('min_child_samples', 20, 100)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True)

    params = {
        'n_estimators': n_estimators, 'learning_rate': learning_rate, 'num_leaves': num_leaves,
        'max_depth': max_depth, 'min_child_samples': min_child_samples, 'subsample': subsample,
        'colsample_bytree': colsample_bytree, 'reg_alpha': reg_alpha, 'reg_lambda': reg_lambda,
        'random_state': RANDOM_SEED, 'n_jobs': -1, 'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    rmse_scores = []
    for train_idx, val_idx in cv.split(X):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        X_train_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, target_col='Neighborhood')
        X_val_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, X_val_fold, target_col='Neighborhood')

        local_preprocessor = preprocessor_template
        local_preprocessor.fit(X_train_fold_te)

        X_train_transformed = local_preprocessor.transform(X_train_fold_te)
        X_val_transformed = local_preprocessor.transform(X_val_fold_te)

        model.fit(X_train_transformed, y_train_fold,
                  eval_set=[(X_val_transformed, y_val_fold)],
                  eval_metric='rmse',
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        y_pred = model.predict(X_val_transformed)
        rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))
        rmse_scores.append(rmse)
    return np.mean(rmse_scores)

print("Creating Optuna study for LightGBM with TPESampler...")
study_lgbm = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_SEED))
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)
best_lgbm_params = study_lgbm.best_params
print(f"Final best LightGBM parameters: {best_lgbm_params}")

# Step 10: Hyperparameter tuning for RandomForest with Optuna
print("\nTuning RandomForest hyperparameters with Optuna...")
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 200, 1500, step=100)
    max_depth = trial.suggest_int('max_depth', 5, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_float('max_features', 0.5, 1.0)
    
    params = {
        'n_estimators': n_estimators, 'max_depth': max_depth, 'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf, 'max_features': max_features,
        'random_state': RANDOM_SEED, 'n_jobs': -1
    }
    model = RandomForestRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    rmse_scores = []
    for train_idx, val_idx in cv.split(X):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        X_train_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, target_col='Neighborhood')
        X_val_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, X_val_fold, target_col='Neighborhood')

        local_preprocessor = preprocessor_template
        local_preprocessor.fit(X_train_fold_te)

        X_train_transformed = local_preprocessor.transform(X_train_fold_te)
        X_val_transformed = local_preprocessor.transform(X_val_fold_te)

        model.fit(X_train_transformed, y_train_fold)
        y_pred = model.predict(X_val_transformed)
        rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))
        rmse_scores.append(rmse)
    return np.mean(rmse_scores)

print("Creating Optuna study for RandomForest with TPESampler...")
study_rf = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_SEED))
study_rf.optimize(objective_rf, n_trials=30, show_progress_bar=True)
best_rf_params = study_rf.best_params
print(f"Final best RandomForest parameters: {best_rf_params}")


# Step 11: Hyperparameter tuning for CatBoost with Optuna
print("\nTuning CatBoost hyperparameters with Optuna...")
def objective_catboost(trial):
    iterations = trial.suggest_int('iterations', 500, 2000, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1, log=True)
    depth = trial.suggest_int('depth', 4, 10)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1e-4, 10.0, log=True)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.6, 1.0)
    
    params = {
        'iterations': iterations, 'learning_rate': learning_rate, 'depth': depth,
        'l2_leaf_reg': l2_leaf_reg, 'subsample': subsample, 'colsample_bylevel': colsample_bylevel,
        'random_seed': RANDOM_SEED, 'verbose': 0, 'loss_function': 'RMSE', 'eval_metric': 'RMSE',
        'thread_count': -1
    }
    model = CatBoostRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    rmse_scores = []
    for train_idx, val_idx in cv.split(X):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        X_train_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, target_col='Neighborhood')
        X_val_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, X_val_fold, target_col='Neighborhood')

        local_preprocessor = preprocessor_template
        local_preprocessor.fit(X_train_fold_te)

        X_train_transformed = local_preprocessor.transform(X_train_fold_te)
        X_val_transformed = local_preprocessor.transform(X_val_fold_te)

        model.fit(X_train_transformed, y_train_fold,
                  eval_set=[(X_val_transformed, y_val_fold)],
                  early_stopping_rounds=50, verbose=0)

        y_pred = model.predict(X_val_transformed)
        rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))
        rmse_scores.append(rmse)
    return np.mean(rmse_scores)

print("Creating Optuna study for CatBoost with TPESampler...")
study_catboost = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_SEED))
study_catboost.optimize(objective_catboost, n_trials=30, show_progress_bar=True)
best_catboost_params = study_catboost.best_params
print(f"Final best CatBoost parameters: {best_catboost_params}")


# --- NEW: Step 11.5: Hyperparameter tuning for Neural Network (THOR) with Optuna ---
print("\nTuning Neural Network (THOR) hyperparameters with Optuna...")

# Define the PyTorch Neural Network model
class SimpleNN(nn.Module):
    def __init__(self, num_features, hidden_size1=128, hidden_size2=64, dropout_rate=0.5):
        super().__init__()
        self.fc1 = nn.Linear(num_features, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        self.output = nn.Linear(hidden_size2, 1)

    def forward(self, X):
        X = self.dropout1(self.relu1(self.fc1(X)))
        X = self.dropout2(self.relu2(self.fc2(X)))
        return self.output(X)

def objective_nn(trial):
    # Determine the number of features after preprocessing
    # This requires a dummy run or pre-transformation outside the objective
    # For now, we will estimate based on the known features and OHE expansion.
    # A safer approach for exact feature count is to run preprocessor on X once:
    dummy_X_transformed = preprocessor_template.fit_transform(X_te_full[:1])
    num_features = dummy_X_transformed.shape[1]

    hidden_size1 = trial.suggest_int('hidden_size1', 64, 256, step=32)
    hidden_size2 = trial.suggest_int('hidden_size2', 32, 128, step=16)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    max_epochs = trial.suggest_int('max_epochs', 50, 200, step=25)

    model_nn = NeuralNetRegressor(
        module=SimpleNN,
        module__num_features=num_features, # Pass dynamic feature count
        module__hidden_size1=hidden_size1,
        module__hidden_size2=hidden_size2,
        module__dropout_rate=dropout_rate,
        lr=lr,
        max_epochs=max_epochs,
        batch_size=batch_size,
        criterion=nn.MSELoss,
        optimizer=optim.Adam,
        iterator_train__shuffle=True,
        callbacks=[('early_stopping', NeuralNetRegressor.get_default_callbacks()[0])], # Use default early stopping
        verbose=0, # Suppress verbose output
        train_split=None, # Managed by KFold
        random_state=RANDOM_SEED
    )
    
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    rmse_scores = []
    
    for train_idx, val_idx in cv.split(X):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        X_train_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, target_col='Neighborhood')
        X_val_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, X_val_fold, target_col='Neighborhood')

        local_preprocessor = preprocessor_template
        local_preprocessor.fit(X_train_fold_te)

        X_train_transformed = local_preprocessor.transform(X_train_fold_te).astype(np.float32) # Skorch expects float32
        X_val_transformed = local_preprocessor.transform(X_val_fold_te).astype(np.float32)

        model_nn.fit(X_train_transformed, y_train_fold.values.reshape(-1, 1).astype(np.float32)) # Skorch expects 2D target

        y_pred = model_nn.predict(X_val_transformed)
        rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred.flatten())) # Flatten predictions for RMSE
        rmse_scores.append(rmse)
    print(f"NN (THOR) Trial {trial.number} completed with mean RMSE: {np.mean(rmse_scores):.5f}")
    return np.mean(rmse_scores)

print("Creating Optuna study for Neural Network (THOR) with TPESampler...")
study_nn = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_SEED))
study_nn.optimize(objective_nn, n_trials=20, show_progress_bar=True) # Fewer trials for NN as it's typically slower
best_nn_params = study_nn.best_params
print(f"Final best Neural Network (THOR) parameters: {best_nn_params}")
# --- END NEW: Step 11.5 ---


# Step 12: Define and train final individual models
print("\nDefining and training final individual models (XGBoost, LightGBM, RandomForest, CatBoost, Neural Network)...")

xgb_final = xgb.XGBRegressor(**best_xgb_params, random_state=RANDOM_SEED, eval_metric='rmse') 
lgbm_final = lgb.LGBMRegressor(**best_lgbm_params, random_state=RANDOM_SEED)
rf_final = RandomForestRegressor(**best_rf_params, random_state=RANDOM_SEED)
catboost_final = CatBoostRegressor(**best_catboost_params, random_seed=RANDOM_SEED, verbose=0) 

# NEW: Neural Network (THOR) final model
dummy_X_transformed = preprocessor_template.fit_transform(X_te_full[:1]) # Need exact feature count for final NN
num_features_final_nn = dummy_X_transformed.shape[1]

nn_final = NeuralNetRegressor(
    module=SimpleNN,
    module__num_features=num_features_final_nn,
    module__hidden_size1=best_nn_params['hidden_size1'],
    module__hidden_size2=best_nn_params['hidden_size2'],
    module__dropout_rate=best_nn_params['dropout_rate'],
    lr=best_nn_params['lr'],
    max_epochs=best_nn_params['max_epochs'],
    batch_size=best_nn_params['batch_size'],
    criterion=nn.MSELoss,
    optimizer=optim.Adam,
    iterator_train__shuffle=True,
    verbose=0, # No verbose output during final training
    random_state=RANDOM_SEED
)


xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_template), 
    ('model', xgb_final)
])
lgbm_pipeline = Pipeline([
    ('preprocessor', preprocessor_template), 
    ('model', lgbm_final)
])
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_template), 
    ('model', rf_final)
])
catboost_pipeline = Pipeline([
    ('preprocessor', preprocessor_template), 
    ('model', catboost_final)
])
# NEW: THOR (NN) pipeline
nn_pipeline = Pipeline([
    ('preprocessor', preprocessor_template),
    ('model', nn_final)
])


# Apply target encoding to the full X for final training
X_te_full = apply_kfold_target_encoding(X, y, target_col='Neighborhood')

# Fit pipelines on full training data
print("Training XGBoost pipeline...")
xgb_pipeline.fit(X_te_full, y) 
print("Training LightGBM pipeline...")
lgbm_pipeline.fit(X_te_full, y)
print("Training RandomForest pipeline...")
rf_pipeline.fit(X_te_full, y)
print("Training CatBoost pipeline...")
catboost_pipeline.fit(X_te_full, y)
# NEW: Training THOR (NN) pipeline
print("Training Neural Network (THOR) pipeline...")
# Skorch needs float32 numpy arrays for inputs and 2D for targets
X_te_full_transformed_for_nn = preprocessor_template.transform(X_te_full).astype(np.float32)
y_nn_target = y.values.reshape(-1, 1).astype(np.float32)
nn_pipeline.named_steps['model'].fit(X_te_full_transformed_for_nn, y_nn_target) # Fit skorch model directly

print("Individual models trained.")


# Step 13: Calculate CV metrics for individual models and Ensemble
print("\nCalculating CV metrics for individual models and Ensemble...")

def calculate_cv_metrics(pipeline, X_original, y_original, cv):
    metrics = {'RMSE': [], 'MAE': [], 'R2': [], 'MAPE': []}
    oof_preds = np.zeros(len(X_original))

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_original)):
        X_train_fold, X_val_fold = X_original.iloc[train_idx], X_original.iloc[val_idx]
        y_train_fold, y_val_fold = y_original.iloc[train_idx], y_original.iloc[val_idx]

        X_train_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, target_col='Neighborhood')
        X_val_fold_te = apply_kfold_target_encoding(X_train_fold, y_train_fold, X_val_fold, target_col='Neighborhood')

        local_preprocessor = preprocessor_template
        local_preprocessor.fit(X_train_fold_te)

        X_train_transformed = local_preprocessor.transform(X_train_fold_te)
        X_val_transformed = local_preprocessor.transform(X_val_fold_te)
        
        model_instance = pipeline.named_steps['model'].__class__(**pipeline.named_steps['model'].get_params())
        
        if isinstance(model_instance, xgb.XGBRegressor):
            model_instance.fit(X_train_transformed, y_train_fold, 
                               eval_set=[(X_val_transformed, y_val_fold)], 
                               verbose=False)
        elif isinstance(model_instance, lgb.LGBMRegressor):
            model_instance.fit(X_train_transformed, y_train_fold, 
                               eval_set=[(X_val_transformed, y_val_fold)], 
                               eval_metric='rmse',
                               callbacks=[lgb.early_stopping(50, verbose=False)])
        elif isinstance(model_instance, CatBoostRegressor):
             model_instance.fit(X_train_transformed, y_train_fold,
                               eval_set=[(X_val_transformed, y_val_fold)],
                               early_stopping_rounds=50,
                               verbose=0)
        # NEW: Neural Network specific fit for CV
        elif isinstance(model_instance, NeuralNetRegressor):
            # Skorch needs float32 numpy arrays for inputs and 2D for targets
            X_train_transformed_nn = X_train_transformed.astype(np.float32)
            y_train_fold_nn = y_train_fold.values.reshape(-1, 1).astype(np.float32)
            X_val_transformed_nn = X_val_transformed.astype(np.float32)
            y_val_fold_nn = y_val_fold.values.reshape(-1, 1).astype(np.float32)

            model_instance.fit(X_train_transformed_nn, y_train_fold_nn, 
                               X_test=X_val_transformed_nn, y_test=y_val_fold_nn) # Skorch uses X_test/y_test for validation

            y_pred = model_instance.predict(X_val_transformed_nn).flatten() # Flatten NN output for RMSE
        else: # For RandomForest
            model_instance.fit(X_train_transformed, y_train_fold)
            y_pred = model_instance.predict(X_val_transformed)

        # For non-NN models
        if not isinstance(model_instance, NeuralNetRegressor):
            y_pred = model_instance.predict(X_val_transformed)

        oof_preds[val_idx] = y_pred

        metrics['RMSE'].append(np.sqrt(mean_squared_error(y_val_fold, y_pred)))
        metrics['MAE'].append(mean_absolute_error(y_val_fold, y_pred))
        metrics['R2'].append(r2_score(y_val_fold, y_pred))
        metrics['MAPE'].append(mean_absolute_percentage_error(y_val_fold, y_pred))
    
    avg_metrics = {k: np.mean(v) for k, v in metrics.items()}
    return avg_metrics, oof_preds

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Get OOF predictions and metrics for each model
print("Getting OOF predictions for XGBoost...")
xgb_metrics, xgb_oof_preds = calculate_cv_metrics(xgb_pipeline, X, y, cv)
print("\nFinal XGBoost CV Metrics:")
for metric, value in xgb_metrics.items():
    print(f"{metric}: {value:.5f}")

print("\nGetting OOF predictions for LightGBM...")
lgbm_metrics, lgbm_oof_preds = calculate_cv_metrics(lgbm_pipeline, X, y, cv)
print("\nFinal LightGBM CV Metrics:")
for metric, value in lgbm_metrics.items():
    print(f"{metric}: {value:.5f}")

print("\nGetting OOF predictions for RandomForest...")
rf_metrics, rf_oof_preds = calculate_cv_metrics(rf_pipeline, X, y, cv)
print("\nFinal RandomForest CV Metrics:")
for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.5f}")

print("\nGetting OOF predictions for CatBoost...")
catboost_metrics, catboost_oof_preds = calculate_cv_metrics(catboost_pipeline, X, y, cv)
print("\nFinal CatBoost CV Metrics:")
for metric, value in catboost_metrics.items():
    print(f"{metric}: {value:.5f}")

# NEW: Get OOF predictions for Neural Network (THOR)
print("\nGetting OOF predictions for Neural Network (THOR)...")
nn_metrics, nn_oof_preds = calculate_cv_metrics(nn_pipeline, X, y, cv)
print("\nFinal Neural Network (THOR) CV Metrics:")
for metric, value in nn_metrics.items():
    print(f"{metric}: {value:.5f}")


# Step 13.5: Optimize Ensemble Weights
print("\nOptimizing ensemble weights using SciPy's minimize...")

# Add NN OOF predictions to the array
oof_preds_array = np.array([xgb_oof_preds, lgbm_oof_preds, rf_oof_preds, catboost_oof_preds, nn_oof_preds])
# Initial weights now for 5 models. A common starting point is equal weights.
initial_weights = np.array([0.20, 0.20, 0.20, 0.20, 0.20]) 


def ensemble_rmse_objective(weights, oof_predictions, true_labels):
    weighted_predictions = np.dot(weights, oof_predictions)
    return np.sqrt(mean_squared_error(true_labels, weighted_predictions))

cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds = [(0.0, 1.0) for _ in range(len(initial_weights))]

result = minimize(ensemble_rmse_objective, initial_weights, args=(oof_preds_array, y),
                  method='SLSQP', bounds=bounds, constraints=cons, options={'disp': False})

optimized_weights = result.x
print(f"Optimized ensemble weights: {optimized_weights}")

ensemble_oof_preds = np.dot(optimized_weights, oof_preds_array)

ensemble_rmse = np.sqrt(mean_squared_error(y, ensemble_oof_preds))
ensemble_mae = mean_absolute_error(y, ensemble_oof_preds)
ensemble_r2 = r2_score(y, ensemble_oof_preds)
ensemble_mape = mean_absolute_percentage_error(y, ensemble_oof_preds)

print("\nFinal Ensemble (Optimized Weighted Average) CV Metrics:")
print(f"RMSE: {ensemble_rmse:.5f}")
print(f"MAE: {ensemble_mae:.5f}")
print(f"R2: {ensemble_r2:.5f}")
print(f"MAPE: {ensemble_mape:.5f}")


# Step 14: Train on full data and predict on test set using the ensemble
print("\nTraining final individual models on full data and generating ensemble predictions for test set...")

test_te = apply_kfold_target_encoding(X, y, X_apply=test, target_col='Neighborhood')

pred_xgb_log = xgb_pipeline.predict(test_te)
pred_lgbm_log = lgbm_pipeline.predict(test_te)
pred_rf_log = rf_pipeline.predict(test_te)
pred_catboost_log = catboost_pipeline.predict(test_te)
# NEW: THOR (NN) prediction on test set
# Skorch expects float32 numpy arrays
test_transformed_for_nn = preprocessor_template.transform(test_te).astype(np.float32)
pred_nn_log = nn_pipeline.named_steps['model'].predict(test_transformed_for_nn).flatten() # Flatten output


# Ensemble prediction using optimized weights
final_pred_log = (optimized_weights[0] * pred_xgb_log +
                  optimized_weights[1] * pred_lgbm_log +
                  optimized_weights[2] * pred_rf_log +
                  optimized_weights[3] * pred_catboost_log +
                  optimized_weights[4] * pred_nn_log) # NEW: Include NN prediction

final_pred = np.expm1(final_pred_log)

print("Ensemble predictions generated.")

# Step 15: Create submission file
SUBMISSION_PATH_ENSEMBLE = '/kaggle/working/AVENGERS_ENSEMBLE.csv'
submission_ensemble = pd.DataFrame({'Id': test_ids, 'SalePrice': final_pred})
submission_ensemble.to_csv(SUBMISSION_PATH_ENSEMBLE, index=False)
print(f"Ensemble submission file created: {SUBMISSION_PATH_ENSEMBLE}")

SUBMISSION_PATH_XGB = '/kaggle/working/JARVIS.csv'
submission_xgb = pd.DataFrame({'Id': test_ids, 'SalePrice': np.expm1(pred_xgb_log)})
submission_xgb.to_csv(SUBMISSION_PATH_XGB, index=False)
print(f"XGBoost (JARVIS) submission file created: {SUBMISSION_PATH_XGB}")

SUBMISSION_PATH_LGBM = '/kaggle/working/MJOLNIR.csv'
submission_lgbm = pd.DataFrame({'Id': test_ids, 'SalePrice': np.expm1(pred_lgbm_log)})
submission_lgbm.to_csv(SUBMISSION_PATH_LGBM, index=False)
print(f"LightGBM (MJOLNIR) submission file created: {SUBMISSION_PATH_LGBM}")

SUBMISSION_PATH_RF = '/kaggle/working/VISION.csv'
submission_rf = pd.DataFrame({'Id': test_ids, 'SalePrice': np.expm1(pred_rf_log)})
submission_rf.to_csv(SUBMISSION_PATH_RF, index=False)
print(f"RandomForest (VISION) submission file created: {SUBMISSION_PATH_RF}")

SUBMISSION_PATH_CATBOOST = '/kaggle/working/ULTRON.csv'
submission_catboost = pd.DataFrame({'Id': test_ids, 'SalePrice': np.expm1(pred_catboost_log)})
submission_catboost.to_csv(SUBMISSION_PATH_CATBOOST, index=False)
print(f"CatBoost (ULTRON) submission file created: {SUBMISSION_PATH_CATBOOST}")

# NEW: Save Neural Network (THOR) predictions
SUBMISSION_PATH_NN = '/kaggle/working/THOR.csv'
submission_nn = pd.DataFrame({'Id': test_ids, 'SalePrice': np.expm1(pred_nn_log)})
submission_nn.to_csv(SUBMISSION_PATH_NN, index=False)
print(f"Neural Network (THOR) submission file created: {SUBMISSION_PATH_NN}")

print("All individual and ensemble submission files should now be available in the 'Output' section of your Kaggle Notebook. 🎉")

print("\nScript completed successfully with added ensemble modeling and enhanced explainability!")
